# 02 – Preprocess SD7K (Kaggle / local CPU)
Resize SD7K pairs to 512×512 JPEG (quality 95) and split by document ID.

**Run this on Kaggle**, then save output as a Kaggle Dataset `sd7k-512`.

Expected raw folder layout:
```
sd7k_raw/
  train/input/      ← shadow images (.png or .jpg)
  train/target/     ← shadow-free images (.png or .jpg)
  test/input/
  test/target/
```

In [ ]:
import os, sys, shutil
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import random
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

# Supported input extensions (dataset uses PNG or JPG)
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def glob_images(directory: Path):
    """Return all image files in a directory, safely handling non-existent paths."""
    if not directory or not directory.exists() or not directory.is_dir():
        return []
    return sorted(p for p in directory.iterdir() if p.suffix.lower() in IMG_EXTS)

def get_split_dir(root: Path, split: str, subdir: str) -> Path:
    """Resolve split and subdir case-insensitively (e.g. train/input, Train/Input)."""
    exact = root / split / subdir
    if exact.exists():
        return exact
    if not root.exists():
        return exact
    split_dir = None
    for s in root.iterdir():
        if s.name.lower() == split.lower():
            split_dir = s
            break
    if split_dir and split_dir.exists():
        for sub in split_dir.iterdir():
            if sub.name.lower() in (subdir.lower(), (subdir + 's').lower()):
                return sub
        return split_dir / subdir
    return exact

def find_sd7k_root() -> Path:
    """Auto-detect SD7K dataset root directory on Kaggle, local, or Colab."""
    search_roots = [Path('/kaggle/input'), Path('data'), Path('.')]
    for base in search_roots:
        if not base.exists():
            continue
        candidates = [base]
        try:
            for p in base.rglob('*'):
                if p.is_dir():
                    candidates.append(p)
        except Exception:
            pass
        for c in candidates:
            try:
                for child in c.iterdir():
                    if child.is_dir() and child.name.lower() in ('train', 'training'):
                        for subchild in child.iterdir():
                            if subchild.is_dir() and subchild.name.lower() in ('input', 'inputs', 'shadow'):
                                for img in subchild.iterdir():
                                    if img.suffix.lower() in IMG_EXTS:
                                        return c
            except Exception:
                continue
    return None

# ── Paths ──────────────────────────────────────────────────────────────────
KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    detected = find_sd7k_root()
    if detected:
        SD7K_RAW = detected
        print(f'✅ Auto-detected SD7K dataset at: {SD7K_RAW}')
    else:
        SD7K_RAW = Path('/kaggle/input/sd7k-raw')
        print(f'⚠️ SD7K not auto-detected. Defaulting to: {SD7K_RAW}')
        if Path('/kaggle/input').exists():
            print('Available paths in /kaggle/input:')
            for p in sorted(Path('/kaggle/input').rglob('*'))[:30]:
                print(f'   {p}')
    OUT_DIR = Path('/kaggle/working/sd7k_512')
else:
    SD7K_RAW = Path('data/sd7k_raw')
    OUT_DIR  = Path('data/sd7k_512')

IMAGE_SIZE = 512
JPEG_Q     = 95
VAL_FRAC   = 0.05
SEED       = 42

print(f'SD7K raw: {SD7K_RAW}')
print(f'Output:   {OUT_DIR}')

In [ ]:
# ── Inspect folder structure ───────────────────────────────────────────────
found_any = False
for split in ['train', 'test']:
    for subdir in ['input', 'target']:
        p = get_split_dir(SD7K_RAW, split, subdir)
        n = len(glob_images(p))
        if n > 0:
            found_any = True
        print(f'  {split}/{subdir}: {n} images (checked: {p})')

train_in = get_split_dir(SD7K_RAW, 'train', 'input')
sample_files = glob_images(train_in)[:5]
print('\nSample filenames:', [f.name for f in sample_files])
if sample_files:
    print(f'Extension detected: {sample_files[0].suffix}')
elif not found_any:
    print('\n❌ WARNING: No images found at expected paths.')
    print('Listing contents of /kaggle/input:')
    if Path('/kaggle/input').exists():
        for p in sorted(Path('/kaggle/input').rglob('*'))[:30]:
            print(f'   {p}')

In [ ]:
# ── Extract document IDs from filenames ───────────────────────────────────
# IMPORTANT: Split by document ID, not by image, to avoid data leakage.
import re

def get_doc_id(filename: str) -> str:
    """Extract document ID from filename.
    
    Common SD7K formats:
      '0001_001.png'     → doc_id = '0001'
      'doc001_cap01.png' → doc_id = 'doc001'
    Adjust the pattern to match your download.
    """
    m = re.match(r'^(.+?)_', filename)
    return m.group(1) if m else filename

train_input_dir = get_split_dir(SD7K_RAW, 'train', 'input')
all_files = [p.name for p in glob_images(train_input_dir)]
if not all_files:
    raise FileNotFoundError(
        f"No images found in {train_input_dir}!\n"
        f"Please check that your dataset is attached on Kaggle and verify its folder structure."
    )
doc_ids = sorted(set(get_doc_id(f) for f in all_files))
print(f'Found {len(all_files)} images, {len(doc_ids)} unique document IDs in training set')
print(f'First 10 doc IDs: {doc_ids[:10]}')

In [ ]:
# ── Train/Val split by document ID ────────────────────────────────────────
random.seed(SEED)
n_val_docs = max(1, int(len(doc_ids) * VAL_FRAC))
val_docs = set(random.sample(doc_ids, n_val_docs))
train_docs = set(doc_ids) - val_docs

print(f'Train docs: {len(train_docs)}, Val docs: {len(val_docs)}')

train_files = [f for f in all_files if get_doc_id(f) in train_docs]
val_files   = [f for f in all_files if get_doc_id(f) in val_docs]
print(f'Train images: {len(train_files)}, Val images: {len(val_files)}')

In [ ]:
# ── Resize and save (always output as .jpg for uniform output dataset) ────
def resize_pair(args):
    fn, src_split, dst_split, src_root, dst_root = args
    src_name = Path(fn)
    dst_name = src_name.stem + '.jpg'  # always save as JPEG
    for subdir in ['input', 'target']:
        src_dir = get_split_dir(src_root, src_split, subdir)
        src = src_dir / src_name
        dst = dst_root / dst_split / subdir / dst_name
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            return
        with Image.open(src).convert('RGB') as img:
            img_resized = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
            img_resized.save(dst, 'JPEG', quality=JPEG_Q)

tasks = []
# Training split
tasks += [(fn, 'train', 'train', SD7K_RAW, OUT_DIR) for fn in train_files]
# Validation split
tasks += [(fn, 'train', 'val',   SD7K_RAW, OUT_DIR) for fn in val_files]
# Official test set (keep as-is)
test_in = get_split_dir(SD7K_RAW, 'test', 'input')
test_files = [p.name for p in glob_images(test_in)]
tasks += [(fn, 'test', 'test',   SD7K_RAW, OUT_DIR) for fn in test_files]

print(f'Total pairs to process: {len(tasks)}')

with ProcessPoolExecutor(max_workers=4) as ex:
    list(tqdm(ex.map(resize_pair, tasks), total=len(tasks), desc='Resizing'))

print('Done! Folder sizes:')
for split in ['train', 'val', 'test']:
    n = len(list((OUT_DIR / split / 'input').glob('*.jpg')))
    print(f'  {split}: {n} pairs')

In [ ]:
# ── Quick sanity check ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt
sample = list((OUT_DIR / 'train' / 'input').glob('*.jpg'))[:3]
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for col, sp in enumerate(sample):
    s = Image.open(sp)
    c = Image.open(OUT_DIR / 'train' / 'target' / sp.name)
    axes[0, col].imshow(s); axes[0, col].set_title(sp.name[:12], fontsize=7); axes[0, col].axis('off')
    axes[1, col].imshow(c); axes[1, col].axis('off')
plt.suptitle(f'Preprocessed pairs ({IMAGE_SIZE}×{IMAGE_SIZE})')
plt.tight_layout(); plt.show()

In [ ]:
# ── (Optional) Zip output for easy download / Kaggle dataset creation ────────
if KAGGLE:
    print('Archiving /kaggle/working/sd7k_512 into zip...')
    shutil.make_archive('/kaggle/working/sd7k_512', 'zip', '/kaggle/working/sd7k_512')
    print('Saved: /kaggle/working/sd7k_512.zip')